# 22 — Embedding Extraction (frozen DINOv2)

Thin notebook: it only **imports**, **calls** `src/embeddings.py`, and **displays**.
It embeds the **union** of all figure sets once for each processed size. A figure's embedding does not depend on which set it belongs to, so each set is just a subset of these rows (used in `23`).

**Input:** `processed/<size>/manifest.csv` (from `21`).
**Output:** `<paths.pipeline_root>/embeddings/<tag>/`: `emb_layer{L}_{pooling}.npy`, `metadata.parquet` (`figure_uid`, `aircraft_uid`, `patent_id`), `model_info.json` and `manifest.json`.
**Needs the GPU.** The devices are set by `analysis.devices`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config_loader import load_config
from src import embeddings

cfg = load_config()
for k in ('model', 'layers', 'pooling', 'batch_size', 'devices'):
    print(f'{k:>10}:', cfg['analysis'].get(k))
print('     sizes:', cfg['extraction']['sizes'], '| tag:', cfg['extraction']['tag'])

## Extract
A run is skipped when its saved figure set already matches the processed manifest. Set `FORCE = True` to recompute it anyway.

In [ ]:
FORCE = False
out_dirs = {size: embeddings.run_extraction(cfg, size, force=FORCE) for size in cfg['extraction']['sizes']}

In [ ]:
for size, d in out_dirs.items():
    res = embeddings.load_embeddings(cfg, d)
    print(d.name, len(res['metadata']), 'figures |', {f'L{k[0]}_{k[1]}': v.shape[1] for k, v in sorted(res['arrays'].items())})